# Step 8: the models retrained at 10 m on real Sentinel-2 imagery

Step D of the Sentinel-2 experiment, and the counterpart of step 6. The same seven Finnish Sentinel-2 scenes, the same scoring, now with the detector and classifiers that were retrained on ShipRSImageNet resampled to 10 m (step 7c). The high-resolution detector and the published Finnish model run again alongside, so all three appear in one table.

Detection is scored against the labelled boxes. The labels carry no vessel type, so the classifiers' answers are only tallied.

Needs the runs of step 7c on Drive. The scenes are already on Drive from step 6; the Copernicus login is only needed if they have been deleted. Nothing is trained, so the T4 is plenty.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Settings

In [ ]:
REPO_URL = 'https://github.com/bprtdaniel/vessel-watch.git'
BRANCH = 'main'

RUNS_DIR = '/content/drive/MyDrive/5.Projects/runs'
S2_DIR = '/content/drive/MyDrive/5.Projects/Data/sentinel2_finland'

## Copernicus login (only used if a scene has to be downloaded again)

In [ ]:
import os
from google.colab import userdata

try:
    os.environ['CDSE_USERNAME'] = userdata.get('CDSE_USERNAME')
    os.environ['CDSE_PASSWORD'] = userdata.get('CDSE_PASSWORD')
    print('Login found')
except Exception as error:
    print('No Copernicus login available, continuing with the scenes on Drive:', type(error).__name__)

## Code

In [ ]:
import os, subprocess

REPO_DIR = '/content/vessel-watch'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

# Colab already ships torch, torchvision, pandas, scikit-learn, matplotlib, pyyaml and opencv
%pip install -q --no-deps -e /content/vessel-watch
%pip install -q ultralytics
%cd /content/vessel-watch
!git log --oneline -1

os.environ['VESSELWATCH_RUNS'] = RUNS_DIR

## Run

The first detector listed is the pipeline's own: its detections are what the classifiers are run on.

In [ ]:
!python -m vesselwatch.detect.finland --name step_d --data-dir "$S2_DIR" --detector at_10m="$RUNS_DIR/yolo11s_obb_10m/weights/best.pt" --detector high_res="$RUNS_DIR/yolo11s_obb/weights/best.pt" --reference --drop-class Dock --classifier configs/crops_10m/level1_resnet101.yaml="$RUNS_DIR/crops10m_level1_resnet101/best.pt" --classifier configs/crops_10m/level3_resnet101.yaml="$RUNS_DIR/crops10m_level3_resnet101/best.pt"

## Examples

Labelled vessels in green, each detector's boxes in its own colour. Saved in `RUNS_DIR/sentinel2_step_d_examples/`.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

for path in sorted(Path(RUNS_DIR, 'sentinel2_step_d_examples').glob('*.png'))[:6]:
    print(path.name)
    display(Image(filename=str(path)))